# How the interaction terms are computed

Three ordinary least squares models stand behind the combinatorial figures, and
they do not measure the same thing. This notebook states each formula, loads
what it produced, and counts what came out, so that a number in a figure can be
traced back to the model that generated it.

Every model is fitted one response gene at a time — 1,041 of them — over cells
carrying knockouts, with the guide module indicators as covariates. The source
notebooks live in `E3Ligase/analysisSingle/Notebooks/CombinatorialPerturbations`.

## Setup

In [1]:
library(data.table)

RDS_DIR <- "/home/eraslab1/Projects/E3Ligase/analysisSingle/Notebooks/CombinatorialPerturbations/RDSFiles"
FDR_CUTOFF <- 0.1

`%ni%` <- Negate(`%in%`)

MODULE_OF_K <- c("0" = "M2", "1" = "M3", "2" = "M6", "3" = "M5", "4" = "M1", "5" = "M4")
moduleLabel <- function(x) {
    for (k in names(MODULE_OF_K)) x <- gsub(paste0("K_?", k), MODULE_OF_K[[k]], x)
    x
}

COVARIATES <- c("(Intercept)", "n_genes", "mt_frac", paste0("leiden", 1:9))

# Genes whose coefficient survives FDR correction, corrected within each gene.
significantPerTerm <- function(d) {
    d <- d[d$term %ni% COVARIATES, ]
    d <- data.table(d)
    d[, FDR := p.adjust(p.value, method = "fdr", n = length(p.value)), by = respGene]
    d <- data.frame(d)
    tapply(d$FDR < FDR_CUTOFF, d$term, sum)
}

## Model 1 — interactions between different modules

Source: `02_FitCrossModuleInteractions.ipynb`.

```
y ~ K_0 + K_1 + ... + K_5
    + K_0*K_1 + K_0*K_2 + ... + K_4*K_5      (all 15 pairs)
```

Fitted on cells carrying knockouts from two different modules. Each `K_i:K_j`
coefficient asks whether the two modules together move a gene by more or less
than the sum of their separate effects. This is the only model of the three that
yields interaction terms directly, because the two knockouts sit in different
covariates and so can be multiplied together.

In [2]:
crossModule <- readRDS(file.path(RDS_DIR, "ComboEffects_lm_residuals_withInteractions.rds"))
crossCounts <- significantPerTerm(crossModule)

cat("response genes:", length(unique(crossModule$respGene)), "\n")
cat("terms fitted:  ", length(crossCounts), "\n\n")

interactionCounts <- crossCounts[grep(":", names(crossCounts))]
data.frame(term = moduleLabel(names(interactionCounts)),
           significantGenes = as.integer(interactionCounts),
           row.names = NULL)

response genes: 1041 
terms fitted:   21 



term,significantGenes
<chr>,<int>
M2:M3,83
M2:M6,123
M2:M5,165
M2:M1,48
M2:M4,33
M3:M6,184
M3:M5,210
M3:M1,35
M3:M4,48


## Model 2 — two knockouts from the *same* module, fitted directly

Source: `03_FitSameModuleDoubleEffects.ipynb`.

```
y ~ K_0 + K_1 + ... + K_5
```

Fitted on cells carrying two knockouts from one module. Note what is missing:
there is no interaction term. A variable cannot interact with itself — `K_0*K_0`
is just `K_0` — so this model cannot express within-module non-additivity at all.

`K_0` here is a **main effect**: how far a same-module double knockout shifts the
gene, additive and non-additive parts together. It is not comparable with the
`K_i:K_j` terms above.

In [3]:
sameGroup <- readRDS(file.path(RDS_DIR, "ComboEffects_doublesSameGroup.rds"))
directCounts <- significantPerTerm(sameGroup)

cat("terms fitted:", paste(names(directCounts), collapse = ", "), "\n")
cat("no term contains ':' -->", !any(grepl(":", names(directCounts))), "\n\n")

data.frame(term = moduleLabel(names(directCounts)),
           significantGenes = as.integer(directCounts),
           row.names = NULL)

terms fitted: K_0, K_1, K_2, K_3, K_4, K_5 
no term contains ':' --> TRUE 



term,significantGenes
<chr>,<int>
M2,252
M3,219
M6,163
M5,122
M1,109
M4,40


## Model 3 — two knockouts from the same module, as an interaction

Source: `04_FitSameModuleInteraction_SingleSplit.ipynb`, aggregated by
`..._sameGroupDoublesResample_2.ipynb`.

This is the trick that makes a within-module interaction estimable. The module's
knockout genes are split at random into two halves, and each half becomes its own
covariate, so the two knockouts of a double land in *different* variables:

```
group1 = random half of the module's genes
group2 = the other half
y ~ group1 + group2 + group1*group2
```

The `group1:group2` coefficient is a genuine within-module interaction. Because
the split is arbitrary, it is repeated five times per module with a different
random half each time; each run is FDR corrected on its own, non-significant
coefficients are set to zero, and the five are averaged. A gene ends up non-zero
if it was significant in at least one of the five runs.

Modules 0 to 4 have enough same-module doubles for this; module 5, which is M4,
does not appear.

In [4]:
oneRun <- readRDS(file.path(RDS_DIR, "ComboEffects_doublesResample_KO_0_1.rds"))
cat("a single run of module 0 fits the term:",
    paste(unique(oneRun$term[grep(":", oneRun$term)]), collapse = ", "), "\n")
cat("   -- that is, half of the module interacted with the other half\n\n")

resampled <- readRDS(file.path(RDS_DIR, "ComboEffects_doublesResampleRes.rds"))
rownames(resampled) <- resampled$respGene
resampled$respGene  <- NULL

resampledCounts <- sapply(resampled, function(x) sum(x != 0))
data.frame(term = moduleLabel(names(resampledCounts)),
           significantGenes = as.integer(resampledCounts),
           row.names = NULL)

a single run of module 0 fits the term: K_0_1:K_0_2 
   -- that is, half of the module interacted with the other half



term,significantGenes
<chr>,<int>
M2:M2,161
M3:M3,272
M6:M6,175
M5:M5,301
M1:M1,115


## Which number belongs in which figure

The two same-module columns below are not two estimates of one quantity. Model 2
is a main effect and model 3 is an interaction, so a panel counting interactions
takes model 3, and a panel showing how far a double knockout moves expression
takes model 2.

In [5]:
comparison <- data.frame(
    module      = moduleLabel(paste0("K_", 0:5)),
    mainEffect_sameModuleDouble = as.integer(directCounts[paste0("K_", 0:5)]),
    interaction_withinModule    = as.integer(
        resampledCounts[match(paste0("K", 0:5, ":K", 0:5), names(resampledCounts))]))

comparison

module,mainEffect_sameModuleDouble,interaction_withinModule
<chr>,<int>,<int>
M2,252,161
M3,219,272
M6,163,175
M5,122,301
M1,109,115
M4,40,NA


:::{note}
The same-module bars of the interaction panel in `Figure5_ACD` come from model 3,
which is why M2 reads 161 there and not the 252 of model 2. The figures using
model 2 are `Figure5_B`, whose axes are observed against predicted fold change,
and the control row of the Figure 5D heatmap.
:::